# 🧠 Segmentação Semântica com DeepLabV3 — Identificação de Classes em Imagens de Obra

**Disciplina:** IA Aplicada à Construção Civil  
**Módulo:** Visão Computacional — Segmentação Semântica com Deep Learning

---

## Onde estamos — e por que isto é diferente

Todos os módulos anteriores usavam algoritmos **clássicos** de visão computacional: limiares, gradientes, grafos. Eles operam sobre pixels e suas relações locais — nenhum deles "sabe" o que está vendo.

A segmentação semântica com deep learning muda o paradigma:

| Abordagem | O modelo "sabe" o que é? | Exemplo de saída |
|-----------|-------------------------|------------------|
| Otsu | ❌ — divide por intensidade | "pixels escuros" vs. "pixels claros" |
| Felzenszwalb | ❌ — divide por similaridade | "região A" vs. "região B" |
| Canny | ❌ — detecta mudanças | "borda aqui" |
| **DeepLabV3** | ✅ — classifica semanticamente | **"isto é uma pessoa", "isto é concreto", "isto é vegetação"** |

Cada pixel recebe um **rótulo de classe** — não apenas um número, mas um conceito aprendido de milhões de imagens anotadas.

---

## DeepLabV3 — arquitetura

```
Imagem (H × W × 3)
        │
        ▼
 ┌─────────────────────────────────────────────┐
 │  Backbone: ResNet-101                       │
 │                                             │
 │  Extrai mapas de características (features) │
 │  em múltiplas escalas — arestas, texturas,  │
 │  formas, partes, objetos                    │
 └──────────────────┬──────────────────────────┘
                    │  feature map (H/8 × W/8 × 2048)
                    ▼
 ┌─────────────────────────────────────────────┐
 │  ASPP — Atrous Spatial Pyramid Pooling      │
 │                                             │
 │  Convoluções dilatadas com 4 taxas (r=1,6,  │
 │  12,18) capturam contexto em múltiplas       │
 │  escalas SEM reduzir resolução              │
 └──────────────────┬──────────────────────────┘
                    │
                    ▼
 ┌─────────────────────────────────────────────┐
 │  Classificador por pixel (1×1 conv)         │
 │  21 classes COCO/Pascal VOC                 │
 │  + Upsample bilinear → resolução original   │
 └──────────────────┬──────────────────────────┘
                    │
             Mapa de classes (H × W)
             cada pixel: 0–20
```

### O que é ASPP e por que importa?

Convoluções normais têm campo receptivo fixo. Uma convolução 3×3 só "vê" uma área de 3×3 pixels ao redor de cada ponto. Para segmentar objetos grandes (laje inteira, fachada de prédio), o modelo precisa de contexto amplo.

A **convolução dilatada** (atrous/dilated) insere espaços entre os pesos do filtro, ampliando o campo receptivo sem aumentar o número de parâmetros:

```
Rate r=1 (normal):   [x][x][x]        Campo: 3×3
Rate r=6 (dilatada): [x][ ][ ][ ][ ][x][ ][ ][ ][ ][x]   Campo: 13×13
```

O ASPP aplica 4 taxas em paralelo e concatena — o modelo vê o pixel em 4 escalas de contexto ao mesmo tempo.

---

## Classes PASCAL VOC — as 21 do modelo

O modelo pré-treinado reconhece as 21 classes do dataset PASCAL VOC 2012:

| ID | Classe | Relevante para obra? |
|----|--------|---------------------|
| 0 | background | — |
| 1 | aeroplane | ❌ |
| 2 | bicycle | ❌ |
| 3 | bird | ❌ |
| 4 | boat | ❌ |
| 5 | bottle | ❌ |
| 6 | bus | ❌ |
| 7 | car | ✅ Veículos de obra |
| 8 | cat | ❌ |
| 9 | chair | ❌ |
| 10 | cow | ❌ |
| 11 | dining table | ❌ |
| 12 | dog | ❌ |
| 13 | horse | ❌ |
| 14 | motorbike | ❌ |
| **15** | **person** | ✅ **LGPD — remoção** |
| 16 | potted plant | ❌ |
| 17 | sheep | ❌ |
| 18 | sofa | ❌ |
| 19 | train | ❌ |
| 20 | tv/monitor | ❌ |

> 💡 **Limitação importante:** o modelo **não foi treinado em imagens de construção civil**. Classes como "concreto", "asfalto", "armadura", "fissura" não existem no vocabulário do modelo. Para isso, é necessário **fine-tuning** com dataset anotado da obra — o que é exatamente o que os módulos Label Studio + U-Net preparam.

---

## Estrutura deste notebook

| Seção | O que você vai fazer |
|-------|----------------------|
| 1 | Verificar GPU e testar inferência com imagem sintética |
| 2 | Montar o Google Drive e configurar parâmetros |
| 3 | Processar todas as imagens em lote |
| 4 | Visualizar segmentação completa e classes por imagem |
| 5 | Filtrar classes específicas (pessoas, veículos) |
| 6 | Analisar confiança e comparar modelos (ResNet-50 vs. 101) |

> 💡 **GPU obrigatória:** DeepLabV3-ResNet101 é pesado. No Colab, vá em *Runtime → Change runtime type → GPU (T4)*. A inferência leva ~1–3 s por imagem com GPU vs. 30–60 s com CPU.

---
## Seção 1 — Verificar GPU e testar inferência

### 1a. Verificar disponibilidade de GPU

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    print(f'✅ GPU disponível: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  GPU não encontrada — usando CPU.')
    print('   Vá em Runtime → Change runtime type → GPU (T4) para acelerar.')
    print('   Com CPU, cada imagem levará ~30–60 s.')

### 1b. Carregar modelo e definir paleta de classes

O download dos pesos (~330 MB) ocorre automaticamente na primeira execução e é cacheado no ambiente Colab.

In [ ]:
import torchvision
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Nomes das classes VOC
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat',
    'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'dining table', 'dog', 'horse', 'motorbike', 'person',
    'potted plant', 'sheep', 'sofa', 'train', 'tv/monitor'
]

# ── Paleta de cores Pascal VOC (oficial)
def _voc_palette():
    palette = np.zeros((256, 3), dtype=np.uint8)
    for i in range(256):
        r = g = b = 0
        c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j)
            g |= ((c >> 1) & 1) << (7 - j)
            b |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
        palette[i] = [r, g, b]
    return palette

VOC_PALETTE = _voc_palette()

# ── Pré-processamento padrão ImageNet
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

# ── Carregar modelo
print('🔄 Carregando DeepLabV3-ResNet101...')
model = torchvision.models.segmentation.deeplabv3_resnet101(
    weights=torchvision.models.segmentation.DeepLabV3_ResNet101_Weights.DEFAULT
).to(device).eval()
print(f'✅ Modelo carregado em {device.type.upper()}.')
print(f'   Parâmetros: {sum(p.numel() for p in model.parameters())/1e6:.0f} M')

### 1c. Teste com imagem sintética (sem Drive)

Criamos uma imagem simples com **dois retângulos de cores diferentes** e uma região de céu azul. Isso testa:
- Se o modelo carregou corretamente
- Se a saída tem o shape esperado (H × W com valores 0–20)
- Se o pipeline de colorização e legendas funciona

> ⚠️ O modelo foi treinado em fotos reais — na imagem sintética a segmentação será imprecisa. O teste valida apenas o ambiente, não a qualidade.

In [ ]:
def inferir(img_pil, model, device):
    """
    Roda DeepLabV3 em uma PIL Image.
    Retorna pred (H×W, int) com IDs de classe 0–20.
    """
    tensor = preprocess(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(tensor)['out'][0]       # (21, H, W)
    pred = out.argmax(0).byte().cpu().numpy()   # (H, W)
    return pred


def colorir_pred(pred):
    """Aplica paleta VOC ao mapa de classes."""
    return VOC_PALETTE[pred]   # (H, W, 3) RGB


def legenda_classes(pred, ax):
    """Adiciona legenda com as classes presentes na predição."""
    ids_presentes = np.unique(pred)
    patches = [
        mpatches.Patch(
            color=VOC_PALETTE[i] / 255.0,
            label=f'{i}: {VOC_CLASSES[i]}'
        )
        for i in ids_presentes if i < len(VOC_CLASSES)
    ]
    ax.legend(handles=patches, bbox_to_anchor=(1.01, 1), loc='upper left',
              fontsize=8, framealpha=0.9)


# ── Imagem sintética
h_s, w_s = 300, 400
arr = np.zeros((h_s, w_s, 3), dtype=np.uint8)
arr[:100, :]        = [135, 180, 220]   # "céu" azul
arr[100:200, :200]  = [80, 80, 75]      # superfície escura
arr[100:200, 200:]  = [160, 155, 140]   # superfície clara
arr[200:, :]        = [60, 120, 60]     # vegetação verde
# Ruído de textura
ruido = np.random.normal(0, 8, arr.shape).astype(np.int16)
arr   = np.clip(arr.astype(np.int16) + ruido, 0, 255).astype(np.uint8)

img_sint_pil = Image.fromarray(arr)
pred_sint    = inferir(img_sint_pil, model, device)
seg_sint     = colorir_pred(pred_sint)

fig, eixos = plt.subplots(1, 2, figsize=(14, 5))
eixos[0].imshow(arr)
eixos[0].set_title('Imagem sintética (teste de ambiente)', fontsize=10)
eixos[0].axis('off')

eixos[1].imshow(seg_sint)
eixos[1].set_title('Segmentação semântica (paleta VOC)', fontsize=10)
eixos[1].axis('off')
legenda_classes(pred_sint, eixos[1])

plt.suptitle('✅ Teste de ambiente — DeepLabV3-ResNet101', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

ids_det = np.unique(pred_sint)
classes_det = [f'{i}:{VOC_CLASSES[i]}' for i in ids_det if i < len(VOC_CLASSES)]
print(f'Classes detectadas : {classes_det}')
print(f'Shape da predição  : {pred_sint.shape}')
print('\n✅ Ambiente OK — pode prosseguir para a Seção 2.')

---
## Seção 2 — Montar o Google Drive e configurar

> ⚠️ **Pasta compartilhada?** Se `fotos_obra` não estiver em *Meu Drive*:  
> *Drive → clique direito → Organizar → Adicionar atalho ao Drive*

> 🔒 **LGPD:** toda inferência ocorre localmente no Colab — nenhuma imagem é enviada a servidores externos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado.')

In [ ]:
# ── Caminhos
PASTA_ENTRADA = Path('/content/drive/MyDrive/fotos_obra')
PASTA_SAIDA   = Path('/content/drive/MyDrive/fotos_obra_semantica')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# ── Classes de interesse para obra
# Altere esta lista para focar nas classes que importam para sua análise
CLASSES_INTERESSE = {
    15: 'person',     # LGPD — remoção
    7 : 'car',        # veículos de obra
    6 : 'bus',        # caminhão/ônibus
}

# ── Redimensionamento (0 = sem resize)
# DeepLabV3 aceita qualquer resolução, mas maiores = mais lentas
LARGURA_MAX = 1024

# ── Extensões
EXTENSOES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

if not PASTA_ENTRADA.exists():
    print(f'⛔ Pasta não encontrada: {PASTA_ENTRADA}')
else:
    arquivos = sorted([p for p in PASTA_ENTRADA.iterdir() if p.suffix.lower() in EXTENSOES])
    print(f'📂 Entrada             : {PASTA_ENTRADA}')
    print(f'📁 Saída               : {PASTA_SAIDA}')
    print(f'🤖 Modelo              : DeepLabV3-ResNet101 ({device.type.upper()})')
    print(f'🎯 Classes de interesse: {CLASSES_INTERESSE}')
    print(f'🖼️  {len(arquivos)} imagem(ns) encontrada(s)')

---
## Seção 3 — Processamento em lote

### O que é salvo por imagem

```
fotos_obra_semantica/
  nome_foto_seg.png          ← mapa de segmentação com paleta VOC
  nome_foto_overlay.png      ← segmentação sobreposta à imagem (alpha=0.5)
  nome_foto_pessoa.png       ← máscara binária de pessoas (se detectadas)
  nome_foto_comparativo.png  ← painel 3 vistas + legenda
  metricas_semantica.csv     ← classes detectadas e % de área por imagem
```

In [ ]:
def overlay_segmentacao(img_rgb_np, seg_rgb_np, alpha=0.5):
    """Mistura imagem original com mapa de segmentação."""
    return (alpha * seg_rgb_np + (1 - alpha) * img_rgb_np).astype(np.uint8)


def stats_classes(pred, n_classes=21):
    """Calcula % de área de cada classe presente."""
    total = pred.size
    resultado = {}
    for i in np.unique(pred):
        if i < n_classes:
            pct = round(100 * (pred == i).sum() / total, 2)
            resultado[VOC_CLASSES[i]] = pct
    return resultado


todas_stats = []
erros       = []

for arq in tqdm(arquivos, desc='Segmentação semântica'):
    try:
        img_pil = Image.open(str(arq)).convert('RGB')

        # Redimensionar mantendo proporção
        if LARGURA_MAX > 0 and img_pil.width > LARGURA_MAX:
            r       = LARGURA_MAX / img_pil.width
            img_pil = img_pil.resize(
                (LARGURA_MAX, int(img_pil.height * r)),
                Image.LANCZOS)

        img_np = np.array(img_pil)

        # ── Inferência
        pred   = inferir(img_pil, model, device)
        seg    = colorir_pred(pred)
        ov     = overlay_segmentacao(img_np, seg)

        # ── Métricas de classe
        areas  = stats_classes(pred)
        row    = {'arquivo': arq.name}
        row.update(areas)
        todas_stats.append(row)

        base = arq.stem

        # Segmentação colorida
        Image.fromarray(seg).save(str(PASTA_SAIDA / f'{base}_seg.png'))

        # Overlay
        Image.fromarray(ov).save(str(PASTA_SAIDA / f'{base}_overlay.png'))

        # Máscara de pessoa (se detectada)
        mask_person = (pred == 15).astype(np.uint8) * 255
        if mask_person.any():
            Image.fromarray(mask_person).save(str(PASTA_SAIDA / f'{base}_pessoa.png'))

        # Painel comparativo
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        seg_bgr = cv2.cvtColor(seg,    cv2.COLOR_RGB2BGR)
        ov_bgr  = cv2.cvtColor(ov,     cv2.COLOR_RGB2BGR)

        painel = np.hstack([img_bgr, seg_bgr, ov_bgr])
        cv2.rectangle(painel, (10, 10), (700, 60), (255, 255, 255), -1)
        classes_str = ', '.join([f'{k}:{v:.0f}%' for k, v in areas.items()
                                  if k != 'background' and v > 1])
        cv2.putText(painel,
                    classes_str[:90] if classes_str else 'apenas background',
                    (18, 46), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (20, 20, 20), 2, cv2.LINE_AA)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_comparativo.png'), painel)

    except Exception as e:
        erros.append((arq.name, str(e)))

# ── CSV
df = pd.DataFrame(todas_stats).fillna(0)
df.to_csv(str(PASTA_SAIDA / 'metricas_semantica.csv'), index=False)

print(f'\n✅ {len(todas_stats)} imagem(ns) processada(s)')
if len(df) > 0 and 'person' in df.columns:
    n_com_pessoa = (df['person'] > 0).sum()
    print(f'   Com pessoas detectadas : {n_com_pessoa} ({100*n_com_pessoa/len(df):.0f}%)')
print(f'   Relatório              : {PASTA_SAIDA / "metricas_semantica.csv"}')
if erros:
    print('\n⚠️  Erros:')
    for n, m in erros: print(f'   {n}: {m}')

---
## Seção 4 — Visualização: segmentação completa e legenda de classes

### 4a. Painel completo por imagem

| Vista | O que analisar |
|-------|----------------|
| **Original** | Referência |
| **Segmentação** | Cada cor corresponde a uma classe VOC. Consulte a legenda ao lado. Regiões de concreto, asfalto e vegetação normalmente ficam em `background` (preto) — limitação do modelo pré-treinado |
| **Overlay** | Sobreposição 50%/50% — permite ver onde a classe detectada se sobrepõe ao elemento real |

In [ ]:
MAX_EXIBIR = 4

for arq in arquivos[:MAX_EXIBIR]:
    img_pil = Image.open(str(arq)).convert('RGB')
    if LARGURA_MAX > 0 and img_pil.width > LARGURA_MAX:
        r       = LARGURA_MAX / img_pil.width
        img_pil = img_pil.resize((LARGURA_MAX, int(img_pil.height * r)), Image.LANCZOS)

    img_np = np.array(img_pil)
    pred   = inferir(img_pil, model, device)
    seg    = colorir_pred(pred)
    ov     = overlay_segmentacao(img_np, seg)

    fig, eixos = plt.subplots(1, 3, figsize=(22, 6))
    dados = [
        (img_np, 'Original', None),
        (seg,    'Segmentação semântica\n(paleta VOC)', None),
        (ov,     'Overlay (alpha=0.5)', None),
    ]
    for ax, (im, t, _) in zip(eixos, dados):
        ax.imshow(im)
        ax.set_title(t, fontsize=9)
        ax.axis('off')

    # Legenda de classes presentes
    legenda_classes(pred, eixos[1])

    areas  = stats_classes(pred)
    resumo = ', '.join([f'{k}: {v:.1f}%' for k, v in areas.items()
                        if k != 'background' and v > 0.5])
    plt.suptitle(
        f'{arq.name}\n{resumo if resumo else "somente background detectado"}',
        fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## Seção 5 — Filtros por classe: pessoas e veículos

### 5a. Isolamento de pessoas (LGPD)

A máscara de pessoas (classe 15) pode ser usada diretamente como entrada para o pipeline de inpainting do módulo de remoção de pessoas (YOLOv8) — ou como alternativa mais leve quando YOLOv8 não estiver disponível.

**Comparação dos dois métodos:**

| | DeepLabV3 (este módulo) | YOLOv8-seg (módulo anterior) |
|-|------------------------|-----------------------------|
| Tipo de saída | Máscara semântica (todos os pixels de pessoa) | Máscara por instância (cada pessoa separada) |
| Múltiplas pessoas | Fundidas em uma máscara | Separadas por instância |
| Velocidade (GPU) | ~1–2 s/imagem | ~0.5 s/imagem |
| Precisão de borda | Boa | Melhor (polígono) |

In [ ]:
MAX_FILTRO = 4

for arq in arquivos[:MAX_FILTRO]:
    img_pil = Image.open(str(arq)).convert('RGB')
    if LARGURA_MAX > 0 and img_pil.width > LARGURA_MAX:
        r       = LARGURA_MAX / img_pil.width
        img_pil = img_pil.resize((LARGURA_MAX, int(img_pil.height * r)), Image.LANCZOS)

    img_np = np.array(img_pil)
    pred   = inferir(img_pil, model, device)

    # Máscara de classes de interesse
    fig, eixos = plt.subplots(1, len(CLASSES_INTERESSE) + 1, figsize=(6*(len(CLASSES_INTERESSE)+1), 5))

    eixos[0].imshow(img_np)
    eixos[0].set_title('Original', fontsize=9)
    eixos[0].axis('off')

    for idx, (class_id, class_name) in enumerate(CLASSES_INTERESSE.items(), start=1):
        mascara = (pred == class_id).astype(np.uint8)
        pct     = round(100 * mascara.sum() / mascara.size, 2)

        if mascara.any():
            # Destacar classe sobre imagem original
            destaque = img_np.copy()
            cor_classe = VOC_PALETTE[class_id].tolist()
            destaque[mascara == 0] = (np.array(destaque[mascara == 0]) * 0.3).astype(np.uint8)
            eixos[idx].imshow(destaque)
            eixos[idx].set_title(
                f'Classe {class_id}: {class_name}\n{pct:.2f}% da imagem ✅', fontsize=9)
        else:
            eixos[idx].imshow(img_np)
            eixos[idx].set_title(
                f'Classe {class_id}: {class_name}\nnão detectada ⬜', fontsize=9)
        eixos[idx].axis('off')

    plt.suptitle(arq.name, fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 5b. Relatório por classe — quais imagens contêm cada classe de interesse

In [ ]:
colunas_interesse = ['arquivo'] + [VOC_CLASSES[i] for i in CLASSES_INTERESSE.keys()
                                    if VOC_CLASSES[i] in df.columns]
df_exib = df[colunas_interesse].copy().reset_index(drop=True)
df_exib.index += 1

def colorir_pct(val):
    if   val > 10 : return 'background-color: #ffcccc'   # vermelho — presença forte
    elif val > 1  : return 'background-color: #fff3cd'   # amarelo — presença moderada
    elif val > 0  : return 'background-color: #d4edda'   # verde — presença fraca
    return ''

cols_num = [c for c in df_exib.columns if c != 'arquivo']
display(df_exib.style.applymap(colorir_pct, subset=cols_num))

print('\n🔴 Vermelho : > 10% — classe ocupa área significativa')
print('🟡 Amarelo  : 1–10% — presença moderada — verificar visualmente')
print('🟢 Verde    : < 1%  — presença pequena ou falso positivo')

---
## Seção 6 — Comparação de modelos: ResNet-50 vs. ResNet-101

O DeepLabV3 está disponível em duas versões de backbone. Esta seção compara:

| Modelo | Parâmetros | Velocidade (T4 GPU) | Precisão (VOC mIoU) |
|--------|-----------|--------------------|-----------------|
| DeepLabV3-ResNet50 | ~39 M | ~0.6 s/img | 66.4% |
| DeepLabV3-ResNet101 | ~61 M | ~1.2 s/img | 67.4% |

> Para datasets de obra onde nenhum dos dois foi treinado especificamente, a diferença de precisão entre eles é mínima. O ResNet-50 é preferível em produção pelo menor custo computacional.

In [ ]:
import time

if arquivos:
    img_pil = Image.open(str(arquivos[0])).convert('RGB')
    if LARGURA_MAX > 0 and img_pil.width > LARGURA_MAX:
        r       = LARGURA_MAX / img_pil.width
        img_pil = img_pil.resize((LARGURA_MAX, int(img_pil.height * r)), Image.LANCZOS)
    img_np = np.array(img_pil)

    print('🔄 Carregando DeepLabV3-ResNet50...')
    model50 = torchvision.models.segmentation.deeplabv3_resnet50(
        weights=torchvision.models.segmentation.DeepLabV3_ResNet50_Weights.DEFAULT
    ).to(device).eval()

    # Inferência com ambos os modelos
    modelos = [
        ('ResNet-101', model),
        ('ResNet-50',  model50),
    ]

    resultados = []
    for nome, mdl in modelos:
        t0   = time.time()
        pred = inferir(img_pil, mdl, device)
        dt   = time.time() - t0
        seg  = colorir_pred(pred)
        resultados.append((nome, pred, seg, dt))

    fig, eixos = plt.subplots(1, 3, figsize=(21, 6))
    eixos[0].imshow(img_np)
    eixos[0].set_title('Original', fontsize=9)
    eixos[0].axis('off')

    for ax, (nome, pred, seg, dt) in zip(eixos[1:], resultados):
        ax.imshow(seg)
        ax.set_title(f'DeepLabV3-{nome}\n{dt:.2f} s', fontsize=9)
        ax.axis('off')
        legenda_classes(pred, ax)

    plt.suptitle(f'Comparação ResNet-50 × ResNet-101 — {arquivos[0].name}',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

    for nome, pred, seg, dt in resultados:
        classes = [f'{k}:{v:.1f}%' for k, v in stats_classes(pred).items()
                   if k != 'background' and v > 0.5]
        print(f'{nome:12} : {dt:.2f}s  |  {classes}')

    # Liberar memória
    del model50
    torch.cuda.empty_cache() if device.type == 'cuda' else None

---
## Próximos passos

### Por que o modelo pré-treinado tem limitações em obra?

```
Dataset de treino (Pascal VOC)          Dataset real de obra
──────────────────────────              ──────────────────────
Fotos de animais, veículos,             Concreto, asfalto, armadura,
móveis, pessoas em cenas                argamassa, fissuras, manchas,
cotidianas                              vegetação em fachada, ferrugem

→ Domínios completamente diferentes → "domain shift" → baixa precisão
```

### O caminho para um modelo de obra

```
DeepLabV3 pré-treinado (VOC)
        │
        ▼
Fine-tuning com dataset de obra
  └── Anotações: Label Studio → máscaras PNG por classe
      Classes: concreto / asfalto / fissura / ferrugem / vegetação / pessoa
        │
        ▼
Modelo especializado em construção civil
  └── Avaliação: mIoU por classe (módulo Otsu → baseline)
  └── Exportar para ONNX → deploy em tablet/drone
```

### Referências

- Chen, L-C. et al. (2017). *Rethinking Atrous Convolution for Semantic Image Segmentation*. arXiv:1706.05587 — DeepLabV3 original
- He, K. et al. (2016). *Deep Residual Learning for Image Recognition*. CVPR — ResNet
- Everingham, M. et al. (2010). *The Pascal VOC Challenge*. IJCV, 88(2) — dataset de treino do modelo
- PyTorch docs: [`torchvision.models.segmentation`](https://pytorch.org/vision/stable/models.html#semantic-segmentation)
- LGPD — Lei nº 13.709/2018, Art. 5º, II — imagem como dado pessoal biométrico
